# M4U3 — YOLOv8 Training (Reproducible Colab Notebook)

**Goal:** Cloud-only reproducibility (Google Colab) for YOLOv8 training + metrics + artifacts.

References (official docs):
- Ultralytics YOLO Train mode: https://docs.ultralytics.com/modes/train/
- Ultralytics YOLO Val mode: https://docs.ultralytics.com/modes/val/
- Colab + GitHub: https://colab.research.google.com/github
- Roboflow dataset export: https://docs.roboflow.com/datasets/dataset-versions/exporting-data


In [ ]:
# ===== 0) BASIC SETUP: clone repo + set working directory =====
import os
import sys
import subprocess
from datetime import datetime

# CHANGE THIS to your repo URL
REPO_URL = "https://github.com/AhmadAlRantisi/M4U3-Assignment---Group-06---yolov8-compliance-detection.git"
REPO_DIR = "/content/m4u3_repo"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

# Ensure required folders exist
os.makedirs("results/evidence", exist_ok=True)
os.makedirs("results/curves", exist_ok=True)
os.makedirs("results/metrics", exist_ok=True)
os.makedirs("docs", exist_ok=True)

print("Working directory:", os.getcwd())
print("Timestamp:", datetime.now().isoformat())


In [ ]:
# ===== 1) INSTALL ULTRALYTICS (must be inside notebook for reproducibility) =====
!pip -q install ultralytics

import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Save environment snapshot
!pip freeze > results/metrics/pip_freeze.txt
print("Saved: results/metrics/pip_freeze.txt")


In [ ]:
# ===== 2) CHECK HARDWARE (for reproducibility proof) =====
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. We'll still run a short verification run if needed.")


In [ ]:
# ===== 3) DATASET DOWNLOAD (Roboflow YOLO export) =====
# IMPORTANT:
# - Go to Roboflow → your dataset version → Export → YOLOv8
# - Copy the provided Python snippet and paste it inside the section below.
# - This notebook expects a data.yaml file path at the end.

# --- You MUST set DATA_YAML after downloading your dataset ---
# Example expected structure after download:
# /content/m4u3_repo/datasets/<your_dataset>/data.yaml

DATA_YAML = ""  # TODO: set this to the full path of data.yaml after Roboflow download

print("DATA_YAML currently:", DATA_YAML)
if DATA_YAML == "":
    print("\nSTOP HERE: You must paste your Roboflow export code and set DATA_YAML.")
    print("Roboflow export docs: https://docs.roboflow.com/datasets/dataset-versions/exporting-data")


In [ ]:
# ===== 4) TRAIN YOLOv8 =====
# Choose model size: yolov8n.pt (fast) or yolov8s.pt (better, slower)
from ultralytics import YOLO

MODEL_VARIANT = "yolov8n.pt"  # change to yolov8s.pt if you want

# If GPU is unavailable, do a verification run (5 epochs)
EPOCHS = 30
if not torch.cuda.is_available():
    EPOCHS = 5

BATCH = 16
IMGSZ = 640

print("MODEL_VARIANT:", MODEL_VARIANT)
print("EPOCHS:", EPOCHS)
print("BATCH:", BATCH)
print("IMGSZ:", IMGSZ)

assert DATA_YAML != "", "DATA_YAML is empty. Set it after dataset download."

model = YOLO(MODEL_VARIANT)

# Train
train_results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project="runs",
    name="m4u3_train",
    exist_ok=True
)

print("Training complete.")


In [ ]:
# ===== 5) VALIDATE + SAVE METRICS =====
# This produces Precision/Recall/mAP50/mAP50-95 based on Ultralytics Val mode.
best_pt = "runs/detect/m4u3_train/weights/best.pt"
assert os.path.exists(best_pt), f"Missing weights: {best_pt}"

model = YOLO(best_pt)
val_results = model.val(data=DATA_YAML, project="runs", name="m4u3_val", exist_ok=True)

# Extract key metrics
metrics = {
    "precision": float(val_results.box.mp),
    "recall": float(val_results.box.mr),
    "map50": float(val_results.box.map50),
    "map50_95": float(val_results.box.map)
}

print("Metrics:", metrics)

# Save to file for README copy/paste
import json
with open("results/metrics/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved: results/metrics/metrics.json")


In [ ]:
# ===== 6) COLLECT CURVES / PLOTS INTO results/curves =====
import shutil

train_dir = "runs/detect/m4u3_train"
val_dir = "runs/detect/m4u3_val"

candidates = []
for d in [train_dir, val_dir]:
    if os.path.exists(d):
        for fname in os.listdir(d):
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                candidates.append(os.path.join(d, fname))

for src in candidates:
    dst = os.path.join("results/curves", os.path.basename(src))
    shutil.copy2(src, dst)

print(f"Copied {len(candidates)} image artifacts into results/curves/")
print("Examples:", candidates[:5])


In [ ]:
# ===== 7) PACKAGE OUTPUTS (easy upload back to GitHub) =====
# This creates a zip you can download from Colab and upload to GitHub results/ folders.
!zip -r m4u3_outputs.zip results runs -x "*/wandb/*" "*/.ipynb_checkpoints/*"
print("Created: m4u3_outputs.zip")
print("Next: Download this zip from Colab and upload evidence/curves/metrics into the repo.")


## Next steps (after this notebook works)

1) Upload evidence screenshots into `results/evidence/` (3–5 annotations, 10 val preds, 5 new preds)
2) Create `docs/error_analysis.md`
3) Create `docs/governance_checklist.md`
4) Update README with reproduce steps + metrics summary + links
